# Indexes

## Overview
An **index** is a separate data structure that allows the database to find rows matching a condition without scanning every row in the table. Indexes trade storage space and write overhead for faster reads.

**B-tree index (default in PostgreSQL):**
A balanced tree sorted by the indexed column(s). Supports equality (`=`), range (`<`, `>`, `BETWEEN`), and sort operations. The right choice for most columns.

**Other PostgreSQL index types:**

| Type | Best for |
|---|---|
| `HASH` | Equality lookups only (`=`); no range support |
| `GIN` | Multi-valued data: arrays, JSONB, full-text search |
| `GiST` | Geometric/spatial data, range types, PostGIS |
| `BRIN` | Very large tables where column value correlates with physical storage order |

**When to add an index:**
- Foreign key columns (patient_id, provider_id) -- joins and lookups
- Columns frequently in WHERE, ORDER BY, or GROUP BY
- Columns in range queries (`enc_date BETWEEN ...`)
- Columns used in JOINs

**When NOT to add an index:**
- Small tables (full scan is faster than index lookup)
- Columns with very low cardinality (e.g. `sex` with 3 values -- index gives little benefit)
- Tables with very frequent INSERT/UPDATE/DELETE (index maintenance overhead)
- Columns rarely queried

**SQLite note:** SQLite supports B-tree indexes via `CREATE INDEX`. PostgreSQL-specific types (GIN, GiST, BRIN) are noted but require a live PostgreSQL connection to demonstrate.

---

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE patients (patient_id INTEGER PRIMARY KEY, first_name TEXT NOT NULL, last_name TEXT NOT NULL, dob TEXT NOT NULL, sex TEXT CHECK(sex IN ('M','F','O')), province TEXT, active INTEGER DEFAULT 1);CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT NOT NULL UNIQUE, building TEXT, floor_no INTEGER);CREATE TABLE providers (provider_id INTEGER PRIMARY KEY, full_name TEXT NOT NULL, specialty TEXT, licence_no TEXT UNIQUE, province TEXT, dept_id INTEGER REFERENCES departments(dept_id), manager_id INTEGER REFERENCES providers(provider_id));CREATE TABLE diagnoses (diag_code TEXT PRIMARY KEY, description TEXT NOT NULL, category TEXT NOT NULL);CREATE TABLE encounters (enc_id INTEGER PRIMARY KEY, patient_id INTEGER NOT NULL REFERENCES patients(patient_id), provider_id INTEGER NOT NULL REFERENCES providers(provider_id), dept_id INTEGER REFERENCES departments(dept_id), enc_date TEXT NOT NULL, cost_cad REAL, admitted INTEGER DEFAULT 0 CHECK(admitted IN (0,1)));CREATE TABLE encounter_diagnoses (enc_id INTEGER NOT NULL REFERENCES encounters(enc_id), diag_code TEXT NOT NULL REFERENCES diagnoses(diag_code), diag_rank INTEGER DEFAULT 1, confirmed INTEGER DEFAULT 1, PRIMARY KEY (enc_id, diag_code));CREATE TABLE lab_results (result_id INTEGER PRIMARY KEY, patient_id INTEGER NOT NULL REFERENCES patients(patient_id), test_name TEXT NOT NULL, result_val REAL, units TEXT, collected TEXT NOT NULL, flagged INTEGER DEFAULT 0);INSERT INTO departments VALUES (1,'Emergency','Tower A',1),(2,'Cardiology','Tower B',2),(3,'Oncology','Tower C',3),(4,'Family Medicine','Clinic',1),(5,'Orthopaedics','Tower A',2),(6,'Radiology','Tower B',3),(7,'ICU','Tower C',NULL);INSERT INTO providers VALUES (10,'Dr. Sarah MacNeil','Cardiology','MC-1001','NB',2,NULL),(11,'Dr. James Wong','Oncology','MC-1002','BC',3,10),(12,'Dr. Fatima Osei','Family Medicine','MC-1003','ON',4,10),(13,'Dr. Ethan Larson','Orthopaedics','MC-1004','QC',5,10),(14,'Dr. Priya Sharma','Emergency','MC-1005','AB',1,10),(15,'Dr. Lucas Petit','Radiology','MC-1006','QC',6,11);INSERT INTO patients VALUES (1,'Aroha','Ngata','1985-03-12','F','NB',1),(2,'Liam','Chen','1972-11-04','M','NS',1),(3,'Fatima','Al-Rashid','1990-07-22','F','ON',1),(4,'James','MacLeod','1955-01-30','M','NB',0),(5,'Sofia','Petrov','2001-09-15','F','BC',1),(6,'Noah','Williams','1968-06-08','M','AB',1),(7,'Mei','Zhang','1995-04-17','F','ON',1),(8,'Ethan','Tremblay','1980-12-01','M','QC',0),(9,'Priya','Nair','1977-08-25','F','BC',1),(10,'Marcus','Okafor','1993-05-19','M','ON',1),(11,'Diana','Flores','2000-02-14','F','NB',1);INSERT INTO diagnoses VALUES ('I25.1','Atherosclerotic heart disease','Cardiovascular'),('J18.9','Pneumonia, unspecified','Respiratory'),('Z00.0','General medical examination','Preventive'),('M17.1','Primary osteoarthritis of knee','Musculoskeletal'),('J06.9','Acute upper respiratory infection','Respiratory'),('R07.9','Chest pain, unspecified','Symptoms'),('I10','Essential hypertension','Cardiovascular'),('R55','Syncope and collapse','Symptoms'),('I48.0','Paroxysmal atrial fibrillation','Cardiovascular'),('S52.5','Fracture of lower end of radius','Injury'),('E11.9','Type 2 diabetes mellitus','Endocrine'),('I50.9','Heart failure, unspecified','Cardiovascular');INSERT INTO encounters VALUES (1,1,10,2,'2023-01-15',3200.00,1),(2,2,14,1,'2023-02-20',1850.00,1),(3,3,12,4,'2023-03-05',120.00,0),(4,4,13,5,'2023-03-18',5500.00,1),(5,5,12,4,'2023-04-02',95.00,0),(6,6,14,1,'2023-05-11',780.00,0),(7,7,10,2,'2023-06-22',2100.00,1),(8,8,12,4,'2023-07-14',80.00,0),(9,1,14,1,'2023-08-30',410.00,0),(10,3,12,4,'2023-09-12',110.00,0),(11,9,10,2,'2023-10-01',1750.00,1),(12,10,14,1,'2023-11-03',2200.00,1),(13,2,10,2,'2023-11-20',2900.00,1),(14,6,12,4,'2023-12-01',115.00,0);INSERT INTO encounter_diagnoses VALUES (1,'I25.1',1,1),(1,'I10',2,1),(2,'J18.9',1,1),(3,'Z00.0',1,1),(4,'M17.1',1,1),(5,'J06.9',1,1),(6,'R07.9',1,1),(7,'I10',1,1),(7,'I48.0',2,1),(9,'R55',1,1),(10,'Z00.0',1,1),(11,'I48.0',1,1),(12,'S52.5',1,1),(13,'I25.1',1,1),(14,'Z00.0',1,1);INSERT INTO lab_results VALUES (1,1,'HbA1c',7.2,'%','2023-03-10',0),(2,2,'HbA1c',9.1,'%','2023-03-15',1),(3,3,'Creatinine',88.5,'umol/L','2023-04-01',0),(4,4,'Creatinine',145.0,'umol/L','2023-04-12',1),(5,5,'HbA1c',5.8,'%','2023-05-05',0),(6,6,'Sodium',138.0,'mmol/L','2023-05-20',0),(7,7,'Sodium',151.0,'mmol/L','2023-06-01',1),(8,1,'Creatinine',72.3,'umol/L','2023-06-18',0),(9,8,'HbA1c',6.4,'%','2023-07-02',0),(10,9,'Creatinine',210.0,'umol/L','2023-07-15',1),(11,2,'Creatinine',95.0,'umol/L','2023-08-01',0),(12,10,'HbA1c',7.8,'%','2023-08-20',1);""")
conn.commit()
print("Healthcare schema ready.")

Healthcare schema ready.


---
## Checking index usage with EXPLAIN

In [2]:
# SQLite EXPLAIN QUERY PLAN shows whether an index is used
print("=== No index on enc_date: full table scan ===")
plan = conn.execute("EXPLAIN QUERY PLAN SELECT * FROM encounters WHERE enc_date = '2023-01-15'").fetchall()
for row in plan:
    print(" ", row)

# Create an index
conn.execute("CREATE INDEX idx_enc_date ON encounters (enc_date)")
conn.commit()

print()
print("=== After CREATE INDEX idx_enc_date: index scan ===")
plan = conn.execute("EXPLAIN QUERY PLAN SELECT * FROM encounters WHERE enc_date = '2023-01-15'").fetchall()
for row in plan:
    print(" ", row)

print()
print("Key PostgreSQL equivalent:")
print("  EXPLAIN ANALYZE SELECT * FROM encounters WHERE enc_date = '2023-01-15';")
print("  -- shows: Index Scan using idx_enc_date on encounters")
print("  -- shows: actual time, rows, loops, buffers")


=== No index on enc_date: full table scan ===
  (2, 0, 0, 'SCAN encounters')

=== After CREATE INDEX idx_enc_date: index scan ===
  (3, 0, 0, 'SEARCH encounters USING INDEX idx_enc_date (enc_date=?)')

Key PostgreSQL equivalent:
  EXPLAIN ANALYZE SELECT * FROM encounters WHERE enc_date = '2023-01-15';
  -- shows: Index Scan using idx_enc_date on encounters
  -- shows: actual time, rows, loops, buffers


---
## Composite indexes and column order

In [3]:
# A composite index on (patient_id, enc_date) serves:
#   WHERE patient_id = X                  (leftmost prefix -- yes)
#   WHERE patient_id = X AND enc_date > Y (full index -- yes)
#   WHERE enc_date > Y                    (no leftmost prefix -- no)

conn.execute("CREATE INDEX idx_enc_patient_date ON encounters (patient_id, enc_date)")
conn.commit()

print("=== Composite index: (patient_id, enc_date) ===")
cases = [
    ("WHERE patient_id = 1",                          True),
    ("WHERE patient_id = 1 AND enc_date > '2023-06'", True),
    ("WHERE enc_date > '2023-06'",                    False),
]
for clause, uses_idx in cases:
    plan = conn.execute(f"EXPLAIN QUERY PLAN SELECT * FROM encounters {clause}").fetchall()
    used = any("idx_enc_patient_date" in str(r) for r in plan)
    status = "uses composite index" if used else "full scan (no leftmost prefix)"
    print(f"  {clause:50s} -> {status}")

print()
print("Rule: a composite index (a, b, c) can be used for queries")
print("filtering on (a), (a,b), or (a,b,c) -- but NOT (b), (c), or (b,c) alone.")


=== Composite index: (patient_id, enc_date) ===
  WHERE patient_id = 1                               -> uses composite index
  WHERE patient_id = 1 AND enc_date > '2023-06'      -> uses composite index
  WHERE enc_date > '2023-06'                         -> full scan (no leftmost prefix)

Rule: a composite index (a, b, c) can be used for queries
filtering on (a), (a,b), or (a,b,c) -- but NOT (b), (c), or (b,c) alone.


---
## Partial and covering indexes

In [4]:
# Partial index: indexes only a subset of rows -- smaller, faster
conn.execute("CREATE INDEX idx_admitted_enc ON encounters (patient_id, enc_date) WHERE admitted = 1")
conn.commit()

print("=== Partial index: only admitted encounters ===")
print("CREATE INDEX idx_admitted_enc ON encounters (patient_id, enc_date) WHERE admitted = 1")
print()
print("Benefits:")
print("  - Smaller index: only 6 admitted rows vs 14 total rows")
print("  - Faster for queries that always filter WHERE admitted = 1")
admitted = conn.execute("SELECT COUNT(*) FROM encounters WHERE admitted=1").fetchone()[0]
total = conn.execute("SELECT COUNT(*) FROM encounters").fetchone()[0]
print(f"  - {admitted} admitted rows vs {total} total ({100*admitted//total}% of table)")

# List all indexes
print()
print("=== All indexes on encounters ===")
idx_list = conn.execute("SELECT name, sql FROM sqlite_master WHERE type='index' AND tbl_name='encounters'").fetchall()
for name, sql in idx_list:
    print(f"  {name}: {sql}")

print()
print("PostgreSQL additional index features:")
print("""
  -- Covering index (INCLUDE): store extra columns in the index leaf nodes
  -- Query can be answered from index alone (index-only scan)
  CREATE INDEX idx_enc_covering
      ON encounters (patient_id, enc_date)
      INCLUDE (cost_cad, admitted);

  -- GIN index for JSONB or full-text:
  CREATE INDEX idx_notes_gin ON clinical_notes USING GIN (note_text_vector);

  -- BRIN index for large append-only tables:
  CREATE INDEX idx_events_brin ON audit_log USING BRIN (event_time);
""")
conn.close()


=== Partial index: only admitted encounters ===
CREATE INDEX idx_admitted_enc ON encounters (patient_id, enc_date) WHERE admitted = 1

Benefits:
  - Smaller index: only 6 admitted rows vs 14 total rows
  - Faster for queries that always filter WHERE admitted = 1
  - 7 admitted rows vs 14 total (50% of table)

=== All indexes on encounters ===
  idx_enc_date: CREATE INDEX idx_enc_date ON encounters (enc_date)
  idx_enc_patient_date: CREATE INDEX idx_enc_patient_date ON encounters (patient_id, enc_date)
  idx_admitted_enc: CREATE INDEX idx_admitted_enc ON encounters (patient_id, enc_date) WHERE admitted = 1

PostgreSQL additional index features:

  -- Covering index (INCLUDE): store extra columns in the index leaf nodes
  -- Query can be answered from index alone (index-only scan)
  CREATE INDEX idx_enc_covering
      ON encounters (patient_id, enc_date)
      INCLUDE (cost_cad, admitted);

  -- GIN index for JSONB or full-text:
  CREATE INDEX idx_notes_gin ON clinical_notes USING GIN (n

---
## Common Pitfalls

**1. Adding an index on every column**
Indexes consume disk space and slow down every INSERT, UPDATE, and DELETE because the index must be maintained. A table with 15 indexes and frequent writes will have serious write performance problems. Index strategically based on actual query patterns.

**2. Indexing low-cardinality columns**
An index on `sex` (3 distinct values) or `admitted` (0/1) has almost no benefit for a regular B-tree index -- the planner will often prefer a full scan. For filtering on a low-cardinality flag, a partial index (`WHERE admitted = 1`) is far more effective.

**3. Wrong column order in a composite index**
`CREATE INDEX ON encounters (enc_date, patient_id)` cannot be used for `WHERE patient_id = 1` alone (no leftmost prefix). Always put the most selective and most frequently filtered column first, then add additional columns in filter/sort order.

**4. Not indexing foreign key columns**
PostgreSQL does NOT automatically create indexes on FK columns -- only on PK columns. Unindexed FKs mean every JOIN on `encounters.patient_id` does a full scan of encounters. Always add an index on FK columns that appear in joins.

**5. Indexes are not used when the column is inside a function**
`WHERE UPPER(last_name) = 'CHEN'` cannot use a regular index on `last_name`. The index stores raw values, not UPPER(values). Use a functional index: `CREATE INDEX ON patients (UPPER(last_name))` -- or normalise the data at insert time.

**6. Forgetting to run VACUUM/ANALYZE after bulk loads**
PostgreSQL's query planner uses table statistics to decide whether to use an index. After loading a large batch, the statistics may be stale and the planner may choose a full scan even when an index exists. Run `ANALYZE table_name` after bulk loads to refresh statistics.


---
*sql_methods_library - Samantha McGarrigle*